## 04 — Financial EDA: The Cost of Uncertainty
**Project:** Supply Chain DI Engine  
**Question:** How much money is at risk because of the East demand shock?  

**Why it matters:**  
- A demand spike is not just an operational problem — it is a financial one.  
- Every unit we fail to deliver = lost revenue + stockout penalty.  
- Every unit we over-stock = holding cost eating into margin.  

**This notebook connects demand behavior to financial consequences.**

**Key Questions:**
1. Which SKUs have the highest stockout penalty in the East?
2. What is our total Revenue at Risk in the East?
3. Which warehouse-to-region shipping lanes are the most expensive?
4. Which SKUs are most expensive to hold if we over-allocate?

In [1]:
import pandas as pd
import duckdb
import matplotlib.pyplot as plt
import seaborn as sns

# Connect to DuckDB
con = duckdb.connect(database=':memory:')

# Register all four CSVs as SQL tables
con.execute("CREATE VIEW demand_history AS SELECT * FROM read_csv_auto('../data/demand_history.csv')")
con.execute("CREATE VIEW products AS SELECT * FROM read_csv_auto('../data/products.csv')")
con.execute("CREATE VIEW warehouses AS SELECT * FROM read_csv_auto('../data/warehouses.csv')")
con.execute("CREATE VIEW shipping_rates AS SELECT * FROM read_csv_auto('../data/shipping_rates.csv')")

print("All 4 tables registered. SQL environment ready.")

# Quick sanity check
con.sql("SELECT COUNT(*) as total_rows FROM demand_history").df()

All 4 tables registered. SQL environment ready.


,total_rows
0,93600


Now let us look at the **dollars at risk**

If we run out of stock in the EAST region, which SKUs will hurt us the mosdt financially?

In [2]:
east_stockout_risk = con.sql("""
    SELECT
        d.sku_id,
        p.product_name,
        p.category,
        ROUND(AVG(d.demand_units), 2) AS avg_weekly_demand,
        p.stockout_penalty_per_unit,
        p.selling_price,
        ROUND(AVG(d.demand_units) * p.stockout_penalty_per_unit, 2) AS weekly_penalty_at_risk
    FROM demand_history d
    JOIN products p ON d.sku_id = p.sku_id
    WHERE d.region = 'East'
    GROUP BY d.sku_id, p.product_name, p.category,
             p.stockout_penalty_per_unit, p.selling_price
    ORDER BY weekly_penalty_at_risk DESC
    LIMIT 10
""").df()

print("--- Top 10 SKUs by Weekly Stockout Penalty at Risk (East Region) ---")
display(east_stockout_risk)

--- Top 10 SKUs by Weekly Stockout Penalty at Risk (East Region) ---


,sku_id,product_name,category,avg_weekly_demand,stockout_penalty_per_unit,selling_price,weekly_penalty_at_risk
0,SKU_262,Bluetooth Adapter,Electronics,58.75,55.88,168.52,3282.95
1,SKU_066,Digital Thermometer,Electronics,42.91,60.75,174.65,2606.99
2,SKU_271,LED Strip Lights 10ft,Electronics,46.45,55.90,202.04,2596.66
3,SKU_096,USB Flash Drive 64GB,Electronics,51.86,49.02,198.73,2541.97
4,SKU_026,Smart Thermostat,Electronics,60.46,41.63,135.69,2517.01
5,SKU_264,Laptop Sleeve,Electronics,66.42,36.21,104.05,2405.18
6,SKU_121,Phone Gimbal,Electronics,53.54,41.78,130.82,2236.84
7,SKU_257,Smart Display,Electronics,34.29,63.99,195.11,2194.12
8,SKU_260,WiFi Range Extender,Electronics,60.70,35.19,118.56,2136.10
9,SKU_165,VR Headset,Electronics,46.90,45.01,185.43,2111.14


### The "Stockout Penalty" Secret

A stockout penalty is not just the price of the product — it represents the **true cost of failing to meet demand**.

It typically includes:

- **Lost Gross Margin:** The profit we would have earned from the sale  
- **Expedited Shipping Costs:** Emergency logistics to fulfill delayed demand  
- **Customer Lifetime Value (CLV):** The long-term cost of losing a dissatisfied customer  

#### Insight

Even if a product sells for around $20, a stockout penalty of over $1,000 per week reveals that the **real financial impact is far greater than the product price**.

#### Analyst Interpretation

> A stockout is not just a missed sale — it is a compounding financial loss.

This means:

- Some products are **far more expensive to run out of than they appear**
- Inventory decisions must consider **penalty risk, not just demand volume**
- High-penalty SKUs should be prioritized in allocation and safety stock planning

Now let us look at **Identifying the HIGH RISK categories**

In [3]:
category_east_risk = con.sql("""
    SELECT 
        p.category,
        COUNT(DISTINCT p.sku_id) as sku_count,
        ROUND(SUM(d.demand_units * p.stockout_penalty_per_unit) / 104, 2) as avg_weekly_penalty_vol
    FROM demand_history d
    JOIN products p ON d.sku_id = p.sku_id
    WHERE d.region = 'East'
    GROUP BY p.category
    ORDER BY avg_weekly_penalty_vol DESC
""").df()

display(category_east_risk)

,category,sku_count,avg_weekly_penalty_vol
0,Electronics,60,83309.41
1,Apparel,60,60126.90
2,Beauty,60,55972.77
3,Home & Kitchen,60,42290.13
4,Grocery,60,38007.49


**High-Cost Lanes & Margin Eaters**

We are going to join shipping_rates with products to see which warehouse-to-region paths are "Margin Eaters"

In [4]:
# Identify the most expensive shipping lanes per unit
shipping_analysis = con.sql("""
    SELECT 
        s.warehouse_id,
        s.destination_region,
        p.product_name,
        p.weight_kg,
        s.cost_per_unit_per_kg,
        p.selling_price,
        -- Calculate total cost to ship THIS specific product ON this lane
        ROUND(p.weight_kg * s.cost_per_unit_per_kg, 2) as shipping_cost_per_unit,
        -- Calculate what % of the price is eaten by shipping
        ROUND((p.weight_kg * s.cost_per_unit_per_kg) / p.selling_price * 100, 2) as shipping_cost_pct
    FROM shipping_rates s
    CROSS JOIN (SELECT * FROM products LIMIT 10) p 
    WHERE s.destination_region = 'East' 
      AND s.warehouse_id != 'WH_East'
    ORDER BY shipping_cost_pct DESC
""").df()

print("--- The Margin Eaters: Cost to ship to East from OTHER warehouses ---")
display(shipping_analysis.head(10))

--- The Margin Eaters: Cost to ship to East from OTHER warehouses ---


,warehouse_id,destination_region,product_name,weight_kg,cost_per_unit_per_kg,selling_price,shipping_cost_per_unit,shipping_cost_pct
0,WH_West,East,Curtains Blackout,2.75,4.40,29.11,12.10,41.57
1,WH_West,East,Chips Tortilla,2.93,4.40,38.90,12.89,33.14
2,WH_West,East,Baking Soda 2lb,2.64,4.40,43.06,11.62,26.98
3,WH_Central,East,Curtains Blackout,2.75,1.95,29.11,5.36,18.42
4,WH_West,East,Chicken Broth 32oz,0.71,4.40,20.26,3.12,15.42
5,WH_Central,East,Chips Tortilla,2.93,1.95,38.90,5.71,14.69
6,WH_West,East,Scalp Scrub,0.54,4.40,16.87,2.38,14.08
7,WH_Central,East,Baking Soda 2lb,2.64,1.95,43.06,5.15,11.96
8,WH_West,East,Jade Roller,0.24,4.40,12.79,1.06,8.26
9,WH_West,East,Cold Brew Concentrate,0.59,4.40,34.88,2.60,7.44


### Shipping Cost Interpretation

A high shipping cost percentage indicates that fulfilling East demand from a distant warehouse can significantly reduce product margin.

However, expensive shipping is not automatically avoided. The decision depends on whether the shipping cost is lower than the financial penalty of a stockout.

This creates a tradeoff:

- If stockout penalty is greater than emergency shipping cost, fulfillment may still be justified.
- If emergency shipping cost is greater than stockout penalty, the company may prefer not to fulfill from that lane.
- If a product has both high stockout penalty and high emergency shipping cost, it should be pre-positioned closer to demand.

This is especially important for Electronics, where high penalties and potentially high shipping costs make reactive fulfillment risky.

**EAST CRITICAL SKU Score**


This combines:

East demand

stockout penalty

weekly penalty risk

product weight

emergency shipping cost from Central to East

emergency shipping cost from West to East

highest emergency shipping cost %

In [5]:
east_critical_skus = con.sql("""
    WITH east_demand AS (
        SELECT
            d.sku_id,
            AVG(d.demand_units) AS avg_east_weekly_demand
        FROM demand_history d
        WHERE d.region = 'East'
        GROUP BY d.sku_id
    ),

    emergency_shipping AS (
        SELECT
            p.sku_id,
            MAX(
                CASE 
                    WHEN s.warehouse_id != 'WH_East' 
                         AND s.destination_region = 'East'
                    THEN p.weight_kg * s.cost_per_unit_per_kg
                END
            ) AS max_emergency_shipping_cost_per_unit,

            MAX(
                CASE 
                    WHEN s.warehouse_id != 'WH_East' 
                         AND s.destination_region = 'East'
                    THEN (p.weight_kg * s.cost_per_unit_per_kg) / p.selling_price * 100
                END
            ) AS max_emergency_shipping_pct
        FROM products p
        CROSS JOIN shipping_rates s
        GROUP BY p.sku_id
    )

    SELECT
        e.sku_id,
        p.product_name,
        p.category,
        ROUND(e.avg_east_weekly_demand, 2) AS avg_east_weekly_demand,
        p.selling_price,
        p.weight_kg,
        p.stockout_penalty_per_unit,
        ROUND(e.avg_east_weekly_demand * p.stockout_penalty_per_unit, 2) AS weekly_stockout_penalty_at_risk,
        ROUND(es.max_emergency_shipping_cost_per_unit, 2) AS max_emergency_shipping_cost_per_unit,
        ROUND(es.max_emergency_shipping_pct, 2) AS max_emergency_shipping_pct,

        ROUND(
            (e.avg_east_weekly_demand * p.stockout_penalty_per_unit)
            + (es.max_emergency_shipping_cost_per_unit * e.avg_east_weekly_demand),
            2
        ) AS east_critical_sku_score

    FROM east_demand e
    JOIN products p 
        ON e.sku_id = p.sku_id
    JOIN emergency_shipping es 
        ON e.sku_id = es.sku_id
    ORDER BY east_critical_sku_score DESC
    LIMIT 15
""").df()

display(east_critical_skus)

,sku_id,product_name,category,avg_east_weekly_demand,selling_price,weight_kg,stockout_penalty_per_unit,weekly_stockout_penalty_at_risk,max_emergency_shipping_cost_per_unit,max_emergency_shipping_pct,east_critical_sku_score
0,SKU_274,Apple Cider Vinegar,Grocery,272.36,4.52,4.52,0.95,258.74,19.89,440.00,5675.35
1,SKU_248,Sparkling Water 12-Pack,Grocery,259.42,8.14,4.44,2.03,526.63,19.54,240.00,5594.72
2,SKU_153,Energy Drink Sugar-Free 12pk,Grocery,192.36,25.11,4.41,4.09,786.74,19.40,77.28,4519.21
3,SKU_145,Whey Protein Powder,Grocery,138.65,41.65,4.71,10.40,1442.00,20.72,49.76,4315.46
4,SKU_276,Almond Butter,Grocery,163.34,13.55,4.59,2.81,458.98,20.20,149.05,3757.72
5,SKU_296,Coconut Oil Organic,Grocery,214.22,16.40,3.33,2.80,599.82,14.65,89.34,3738.59
6,SKU_262,Bluetooth Adapter,Electronics,58.75,168.52,0.58,55.88,3282.95,2.55,1.51,3432.88
7,SKU_070,Honey Raw Organic,Grocery,145.19,11.61,4.93,1.94,281.67,21.69,186.84,3431.18
8,SKU_101,Hot Sauce Variety,Grocery,152.38,37.25,3.45,6.58,1002.63,15.18,40.75,3315.68
9,SKU_264,Laptop Sleeve,Electronics,66.42,104.05,2.83,36.21,2405.18,12.45,11.97,3232.28


### Notebook 04 Conclusion

This notebook translated demand uncertainty into financial risk.

Key findings:

- Electronics has the highest average weekly stockout penalty exposure in the East.
- Some products are expensive to rescue through emergency shipping from distant warehouses.
- High stockout penalty and high shipping cost create the strongest case for pre-positioning inventory.
- The East Critical SKU Score identifies which products should receive priority in allocation decisions.

#### Decision Intelligence Implication

The optimization engine should not allocate inventory based only on demand volume.

It should consider:

- expected demand
- stockout penalty
- product weight
- shipping cost
- warehouse location
- holding cost

The next step is to move from EDA into uncertainty modeling using Monte Carlo simulation.